In [ ]:
import os
import sys
# Se placer à la racine du projet (un niveau au-dessus de notebooks/)
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from tpot import TPOTRegressor
from AutoML_Tpot.data_processing import data_processing
from AutoML_Tpot.normalize import normalize_predictions_by_month

In [5]:
print(f"Répertoire actuel après retour en arrière : {os.getcwd()}")

Répertoire actuel après retour en arrière : /workspaces/AutoMLTPOT


In [ ]:

# Chemin relatif (fonctionne en local et en Docker)
file_path = os.path.join('data', 'kable_data.xlsx')

# Charger les feuilles Excel
ws = pd.read_excel(file_path, sheet_name="Massive for learning")
ws_test = pd.read_excel(file_path, sheet_name="Massive")

print("ws shape:", ws.shape)
print("ws_test shape:", ws_test.shape)
print("Colonnes ws:", list(ws.columns))
print("Colonnes ws_test:", list(ws_test.columns))

In [ ]:
X, Y = data_processing(ws, ws_test)

# La première colonne de X est la cible (variable d'intérêt)
y_target = pd.to_numeric(pd.Series(X[:, 0]), errors='coerce')
X_features = np.nan_to_num(X[:, 1:].astype(float))

# Filtrer les lignes où la cible est valide
valid_idx = ~np.isnan(y_target)
X_features = X_features[valid_idx]
y_target = y_target[valid_idx].values

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.25, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
reg = TPOTRegressor(verbosity=2, population_size=50, generations=1, random_state=35)
reg.fit(X_train, y_train)
print('Score R² sur test set:', reg.score(X_test, y_test))

# Exporter le pipeline trouvé (chemin relatif)
export_path = os.path.join('python_boston', 'top_bostonkable.py')
reg.export(export_path)
print(f'Pipeline exporté vers {export_path}')

is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor


/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:1230: FutureWarning: passing a class to None is deprecated and will be removed in 1.8. Use an instance of the class instead.
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:1270: FutureWarning: passing a class to None is deprecated and will be removed in 1.8. Use an instance of the class instead.
  warnings.warn(


is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor
is_classifier
is_regressor


Version 0.12.2 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


Optimization Progress:   0%|          | 0/100 [00:00<?, ?pipeline/s]


Generation 1 - Current best internal CV score: -6.381133968764561e-06

Best pipeline: XGBRegressor(input_matrix, learning_rate=0.001, max_depth=4, min_child_weight=7, n_estimators=100, n_jobs=1, objective=reg:squarederror, subsample=0.7500000000000001, verbosity=0)
-6.821028998810857e-06
Exported to python_boston/top_bostonkable.py


In [ ]:
# Données de test (Y vient de data_processing, même colonnes que X_features)
testing_features = np.nan_to_num(Y[:, 1:].astype(float))
print("Shape X_features (train):", X_features.shape)
print("Shape testing_features:", testing_features.shape)

Shape of training features: (1339, 18)
Shape of testing features: (1339, 18)


In [ ]:
# Réentraîner le meilleur pipeline sur tout le jeu d'entraînement
best_pipeline = reg.fitted_pipeline_

best_pipeline.fit(X_train, y_train)

# Évaluation sur le TEST set (pas sur le train !)
predictions_test = best_pipeline.predict(X_test)
predictions_test = np.clip(predictions_test, 0, 1)

mse = mean_squared_error(y_test, predictions_test)
r2 = r2_score(y_test, predictions_test)

print(f"MSE (test): {mse:.6f}")
print(f"R²  (test): {r2:.6f}")

# Prédictions sur les vraies données de test (feuille "Massive")
predictions = np.clip(best_pipeline.predict(testing_features), 0.0001, 1)
print(f"Prédictions shape: {predictions.shape}")

       Statistiques Факт  Statistiques Прогноз
count        1339.000000           1339.000000
mean            0.005478              0.005478
std             0.002544              0.000027
min             0.001009              0.005400
25%             0.003303              0.005458
50%             0.005468              0.005478
75%             0.007637              0.005496
max             0.009997              0.005573


In [ ]:
# Normaliser les prédictions
normalized_results = normalize_predictions_by_month(ws_test, predictions)

# Exporter les résultats (chemin relatif)
output_file_path = os.path.join('python_boston', 'results.xlsx')
normalized_results.to_excel(output_file_path, index=False, engine='openpyxl')

print(f"Résultats normalisés exportés vers {output_file_path}")

Сумма прогнозов по месяцам (полные месяцы) :
Mois
2022-01    1.0
2022-02    1.0
2022-03    1.0
2022-04    1.0
2022-05    1.0
2022-06    1.0
2022-07    1.0
2022-08    1.0
2022-09    1.0
2022-10    1.0
2022-11    1.0
2022-12    1.0
2023-01    1.0
2023-02    1.0
2023-03    1.0
2023-04    1.0
2023-05    1.0
2023-06    1.0
2023-07    1.0
2023-08    1.0
2023-09    1.0
2023-10    1.0
2023-11    1.0
2023-12    1.0
2024-01    1.0
2024-02    1.0
2024-03    1.0
2024-04    1.0
2024-05    1.0
2024-06    1.0
2024-07    1.0
2024-08    1.0
2024-09    1.0
2024-10    1.0
2024-11    1.0
2024-12    1.0
2025-01    1.0
2025-02    1.0
2025-03    1.0
2025-04    1.0
2025-05    1.0
2025-06    1.0
2025-07    1.0
2025-08    1.0
Freq: M, Name: Prédiction, dtype: float32
Les résultats normalisés ont été exportés vers /workspaces/AutoMLTPOT/python_boston/results.xlsx
